# 🧠 MyBabyAI (CodeMind) - Cloud Training System

Bu notebook, CodeMind modellerini **Google Colab** veya **Kaggle** üzerinde eğitmek için tasarlanmıştır.

### Özellikler:
- Colab/Kaggle Otomatik Algılama
- 350M-MoE Optimizasyonu
- LoRA / Full-Training Desteği
- Çoklu Veriseti Havuzu (Turkish/English/Code)
- Otomatik Checkpoint Yönetimi

In [ ]:
# @title 1. 🛠️ SİSTEM VE BAĞIMLILIKLARI KUR
import os, sys

# Ortam Algılama
ENV = "colab" if "google.colab" in sys.modules else "kaggle" if os.path.exists("/kaggle") else "local"
print(f"Detected Environment: {ENV.upper()}")

# Paket Kurulumu
!pip install -q transformers peft bitsandbytes sentencepiece huggingface_hub
!pip install -q pyyaml python-dotenv rich tqdm psutil requests datasets chromadb sentence-transformers

if ENV == "colab":
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None
elif ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except:
        HF_TOKEN = None
else:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    !huggingface-cli login --token $HF_TOKEN
    print("✅ HuggingFace Girişi Yapıldı")


In [ ]:
# @title 2. 📂 REPO VE DİZİN AYARLARI
REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

if not os.path.exists("mybabyai"):
    !git clone -b $BRANCH $REPO_URL
    %cd mybabyai
else:
    %cd mybabyai
    !git pull

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.makedirs("models/fine_tuned", exist_ok=True)
print("✅ Çalışma dizini hazır.")


In [ ]:
# @title 3. ⚙️ EĞİTİM KONFİGÜRASYONU

model_size = "350M-MoE" # @param ["125M", "350M", "350M-MoE", "650M"]
training_mode = "full" # @param ["lora", "full"]
load_from_checkpoint = False # @param {type:"boolean"}

batch_size = 1 # @param {type:"integer"}
gradient_accumulation = 4 # @param {type:"integer"}
learning_rate = 5e-5 # @param {type:"number"}
epochs = 3 # @param {type:"integer"}
save_steps = 200 # @param {type:"integer"}

output_dir = "/content/drive/MyDrive/mybabyai_checkpoints" if ENV == "colab" else "/kaggle/working/checkpoints"

if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(output_dir, exist_ok=True)  # Colab ve Kaggle icin

print(f"--- {model_size} ({training_mode}) Yapılandırması Tamam ---")

In [ ]:
# @title 4. 🤖 MODEL & TRAINER BAŞLATMA
from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
config.set("training.output_dir", output_dir)
config.set("training.per_device_train_batch_size", batch_size)
config.set("training.gradient_accumulation_steps", gradient_accumulation)
config.set("training.learning_rate", learning_rate)
config.set("training.num_train_epochs", epochs)
config.set("training.save_steps", save_steps)

model_manager = ModelManager(config)

if load_from_checkpoint:
    print("📦 Mevcut checkpoint yükleniyor...")
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    print(f"🌱 Sıfırdan CodeMind-{model_size} oluşturuluyor...")
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)
# NOT: prepare_model_for_training, train_from_pool içinde otomatik çağrılır.
# Burada çift çağrı gerekmez.

print(f"✅ {model_manager.model_name} Eğitime Hazır!")


In [ ]:
# @title 5. 🎯 DATASET HAVUZU (TÜRKÇE ODAKLI)

dataset_pool = [
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 5000
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 3000
    }
]

print(f"📊 Dataset havuzu: {len(dataset_pool)} kaynak hazır.")


In [ ]:
# @title 6. 🚀 EĞİTİMİ BAŞLAT
try:
    trainer.train_from_pool(dataset_pool, training_type=training_mode, use_notebook_callback=True)
except KeyboardInterrupt:
    print("\n🛑 Eğitim kullanıcı tarafından durduruldu.")


In [ ]:
# @title 7. 🧪 TEST (INFERENCE)
prompt = "Merhaba, nasılsın?" # @param {type:"string"}
max_new_tokens = 100 # @param {type:"integer"}

print("Generating...")
response = model_manager.generate(prompt, max_new_tokens=max_new_tokens)
print(f"\n🤖 AI: {response}")


## 📖 Yardımcı Rehber

### Checkpoint Dosyalarını Almak:
1. **Colab:** Dosyalar `/content/drive/MyDrive/mybabyai_checkpoints` klasörüne (Google Drive) otomatik kaydedilir.
2. **Kaggle:** Eğitim bitince sağ üstten **"Save Version"** butonuna basıp **"Quick Save"** seçin. Kaydedildikten sonra Notebook sayfasındaki **Output** sekmesinden indirebilirsiniz.

### OOM Hatası Alırsanız:
- `training_mode`'u **lora** yapın.
- `batch_size` 1 olduğundan emin olun.
- Dataset `max_samples` sayılarını düşürün.